# 08 · APIs with `requests`

Until now your programs only knew what you typed in or saved in a file. With an **API** your program can talk to **other computers on the internet**: ask for weather, Pokémon stats, movies, currency rates, and get answers back in seconds.

> ⚠️ **This notebook needs an internet connection**, and it uses `requests`, a library you install with `pip`. (Read the *Developer Tools Guide* for `pip` and virtual environments.) The code cells here are **not pre-run**: run them on your own computer and compare with the **"What you should see"** notes.

**What's inside**

| # | Topic |
|---|---|
| 1 | What is an API? |
| 2 | Installing and importing `requests` |
| 3 | Your first request |
| 4 | Reading JSON: dictionaries inside lists inside dictionaries |
| 5 | Query parameters with `params=` |
| 6 | When things go wrong: status codes and exceptions |
| 7 | Being polite: caching |
| 8 | Project: a mini Pokédex |
| 9 | Practice and next steps |

We will use **[PokeAPI](https://pokeapi.co)**, a free API of Pokémon data. It needs no account or key.

---
## 1. What is an API?

**API** = *Application Programming Interface*: a way for one program to ask another program for something.

> 💡 **Restaurant analogy:** you (your program) sit at a table. The **kitchen** (the server) has the food (the data). You can't walk into the kitchen. You tell the **waiter** (the API) what you want from the **menu** (the list of things you may ask for), and the waiter brings it back.

### How a web API request works
1. Your program sends an **HTTP request** to a **URL** (just like your browser does when you type an address).
2. The server sends back an **HTTP response**: a **status code** plus the **data**, usually in **JSON** (the format you met in the Files notebook).

Anatomy of a URL:

```
https://pokeapi.co/api/v2/pokemon/pikachu
└─┬──┘  └───┬────┘ └──┬──┘ └──┬──┘ └──┬──┘
scheme   domain     base    endpoint  what we want
```

### Request types (methods)
| Method | Meaning |
|---|---|
| `GET` | **fetch** data (what we'll use) |
| `POST` | **send** new data |
| `PUT` / `PATCH` | update data |
| `DELETE` | remove data |

### Status codes: the server's quick answer
| Code | Meaning |
|---|---|
| **200** | OK ✅ |
| **201** | Created |
| **400** | Bad request (you asked wrongly) |
| **401 / 403** | Not logged in / not allowed |
| **404** | Not found ❌ |
| **429** | Too many requests: slow down! |
| **500** | The server itself had a problem |

Rule of thumb: **2xx = success, 4xx = your mistake, 5xx = the server's mistake.**

---
## 2. Installing and importing `requests`

`requests` is not part of Python's standard library, so you must install it once. In a notebook, this magic command installs it into the Python that the notebook is using:

In [ ]:
%pip install requests

In [ ]:
import requests
print(requests.__version__)

*(If you work from a terminal instead, the command is `python -m pip install requests`. Always do this inside a virtual environment. See the Developer Tools Guide.)*

---
## 3. Your first request

`requests.get(url)` sends a GET request and returns a **Response** object. The useful parts:

| Part | Gives you |
|---|---|
| `response.status_code` | the number (200, 404, …) |
| `response.json()` | the data, **already converted into Python** dictionaries and lists |
| `response.text` | the raw text |

In [ ]:
import requests

BASE_URL = "https://pokeapi.co/api/v2"

response = requests.get(f"{BASE_URL}/pokemon/pikachu", timeout=5)
print(response.status_code)

data = response.json()
print(type(data))
print(data["name"])
print(data["id"])

**What you should see**
```
200
<class 'dict'>
pikachu
25
```

> 💡 Always pass `timeout=5` (seconds). Without it, a slow server could make your program wait forever.

---
## 4. Reading JSON

`response.json()` gives you ordinary Python dictionaries and lists, so everything from the Dictionaries notebook applies. The trick is **finding your way** through the nesting. Two helpful habits:

1. Print `data.keys()` to see what's at the top level.
2. Print one piece at a time, going deeper step by step.

In [ ]:
print(list(data.keys())[:8])             # the response has many keys; show a few
print(data["height"], data["weight"])
print(data["types"])                      # a LIST of dictionaries

`types` is a **list of dicts**, and each of those holds *another* dict called `"type"`. So the type name is three steps in: `data["types"][0]["type"]["name"]`. Use a comprehension to collect all type names, and a dictionary comprehension for the stats:

In [ ]:
type_names = [t["type"]["name"] for t in data["types"]]
print(type_names)

stats = {s["stat"]["name"]: s["base_stat"] for s in data["stats"]}
print(stats)
print("Total:", sum(stats.values()))

# PokeAPI stores height in decimetres and weight in hectograms
print("Height:", data["height"] / 10, "m")
print("Weight:", data["weight"] / 10, "kg")

**What you should see** (for Pikachu)
```
['electric']
{'hp': 35, 'attack': 55, 'defense': 40, 'special-attack': 50, 'special-defense': 50, 'speed': 90}
Total: 320
Height: 0.4 m
Weight: 6.0 kg
```

---
## 5. Query parameters with `params=`

Many APIs let you tune a request with **query parameters**, the part of the URL after `?`, like `...?limit=5&offset=10`. You *could* build that text yourself, but it's cleaner (and safer, as special characters get encoded for you) to pass a dictionary to `params=`.

In [ ]:
response = requests.get(
    f"{BASE_URL}/pokemon",
    params={"limit": 5, "offset": 0},
    timeout=5,
)
print(response.url)                      # see the final URL that was really used
page = response.json()

for entry in page["results"]:
    print(entry["name"])

**What you should see**
```
https://pokeapi.co/api/v2/pokemon?limit=5&offset=0
bulbasaur
ivysaur
venusaur
charmander
charmeleon
```

Change `offset` to `5` to get the *next* five (this is called **pagination**).

---
## 6. When things go wrong

Real programs must expect failure: wrong names, no Wi-Fi, a slow server. A request can go wrong in two different ways:

1. **The server answered, but with an error code** (like 404). Python does *not* crash by itself.
2. **The request couldn't be completed at all** (no internet, timeout). `requests` raises an **exception**.

`response.raise_for_status()` turns a bad status code (4xx/5xx) into an exception (`HTTPError`) so that **all** failures can be handled in one `try/except`, using what you learned in the Exceptions notebook.

| Exception | Meaning |
|---|---|
| `requests.exceptions.HTTPError` | the server replied 4xx / 5xx (after `raise_for_status()`) |
| `requests.exceptions.Timeout` | the server took longer than `timeout` |
| `requests.exceptions.ConnectionError` | couldn't reach the server (no internet, wrong domain) |
| `requests.exceptions.RequestException` | **parent of all of them**: a safety net |

Put the specific ones first and the parent last.

In [ ]:
response = requests.get(f"{BASE_URL}/pokemon/not-a-pokemon", timeout=5)
print(response.status_code)             # 404 - but no crash yet!

In [ ]:
def fetch_pokemon(name):
    try:
        response = requests.get(f"{BASE_URL}/pokemon/{name}", timeout=5)
        response.raise_for_status()                  # 404 becomes an exception here
        return response.json()
    except requests.exceptions.HTTPError:
        print(f"No Pokémon called '{name}' (server said {response.status_code}).")
    except requests.exceptions.Timeout:
        print("The server took too long to answer.")
    except requests.exceptions.ConnectionError:
        print("Can't reach the internet or the server.")
    except requests.exceptions.RequestException as e:
        print("Something else went wrong:", e)
    return None


print(fetch_pokemon("ditto") is not None)
print(fetch_pokemon("not-a-pokemon"))

**What you should see**
```
True
No Pokémon called 'not-a-pokemon' (server said 404).
None
```

To test the "no internet" branch, turn off your Wi-Fi and run it again.

---
## 7. Being polite: caching

Free APIs ask users not to hammer them. If you already asked for Pikachu, **don't ask again**: save the answer and reuse it. Saving answers you've already fetched is called **caching**, and you already know how: a dictionary + a JSON file.

```
lookup("pikachu")
   ├─ in the cache?  → return it (no internet needed!)
   └─ not in cache?  → ask the API → save to cache → return it
```

This also makes your program **faster** and lets it work **offline** for things you've seen before.

> 📜 PokeAPI's fair-use rule: *cache what you fetch, and don't send huge numbers of requests.* No account or API key is needed.

---
## 8. Project: a mini Pokédex

Now we combine everything: **classes** (a `Pokemon` object), **`@classmethod`** (build one from JSON), **`@property`** (the stat total), **dictionaries** (the cache), **JSON files**, **exceptions**, a **menu loop**, and **`requests`**.

### Step 1: the `Pokemon` class
Notice how the class turns the messy nested JSON into a clean object, and how `from_json` is a *classmethod*: an alternative constructor.

In [1]:
class Pokemon:
    def __init__(self, id, name, height_m, weight_kg, types, stats):
        self.id = id
        self.name = name
        self.height_m = height_m
        self.weight_kg = weight_kg
        self.types = types
        self.stats = stats

    @classmethod
    def from_json(cls, data):
        """Build a Pokemon from the raw PokeAPI response."""
        types = [t["type"]["name"] for t in data["types"]]
        stats = {s["stat"]["name"]: s["base_stat"] for s in data["stats"]}
        return cls(
            data["id"],
            data["name"],
            data["height"] / 10,          # decimetres -> metres
            data["weight"] / 10,          # hectograms -> kilograms
            types,
            stats,
        )

    @property
    def total(self):
        return sum(self.stats.values())

    def to_dict(self):
        """A plain dictionary, ready to be saved as JSON."""
        return {
            "id": self.id, "name": self.name,
            "height_m": self.height_m, "weight_kg": self.weight_kg,
            "types": self.types, "stats": self.stats,
        }

    def __str__(self):
        lines = [
            f"#{self.id} {self.name.title()}  [{', '.join(self.types)}]",
            f"   Height: {self.height_m} m   Weight: {self.weight_kg} kg",
        ]
        for stat, value in self.stats.items():
            lines.append(f"   {stat:<16}{value:>4}")
        lines.append(f"   {'TOTAL':<16}{self.total:>4}")
        return "\n".join(lines)

### Step 2: the lookup function with caching and error handling

In [ ]:
import requests, json, os

BASE_URL = "https://pokeapi.co/api/v2"
CACHE_FILE = "pokedex_cache.json"


def load_cache():
    try:
        with open(CACHE_FILE) as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def save_cache(cache):
    with open(CACHE_FILE, "w") as f:
        json.dump(cache, f, indent=2)


def lookup(name, cache):
    name = name.strip().lower()

    if name in cache:                                  # 1. already known?
        print("(from cache)")
        return Pokemon(**cache[name])

    try:                                               # 2. ask the API
        response = requests.get(f"{BASE_URL}/pokemon/{name}", timeout=5)
        response.raise_for_status()
    except requests.exceptions.HTTPError:
        print(f"No Pokémon called '{name}'.")
        return None
    except requests.exceptions.Timeout:
        print("The server took too long to answer.")
        return None
    except requests.exceptions.ConnectionError:
        print("Can't reach the internet.")
        return None

    pokemon = Pokemon.from_json(response.json())
    cache[name] = pokemon.to_dict()                    # 3. remember it
    save_cache(cache)
    return pokemon

`Pokemon(**cache[name])` uses the `**` unpacking you met in the Functions notebook: the saved dictionary's keys match the `__init__` parameter names exactly, so a saved dictionary becomes an object in one line.

### Step 3: the menu

In [ ]:
cache = load_cache()

while True:
    print("\n===== Mini Pokédex =====")
    print("1. Look up a Pokémon")
    print("2. Compare two Pokémon")
    print("3. Show everything I've looked up")
    print("4. Exit")

    match input("Choose: "):
        case "1":
            pokemon = lookup(input("Name: "), cache)
            if pokemon:
                print(pokemon)

        case "2":
            first = lookup(input("First name: "), cache)
            second = lookup(input("Second name: "), cache)
            if first and second:
                winner = max(first, second, key=lambda p: p.total)
                print(f"{first.name.title()}: {first.total}  vs  {second.name.title()}: {second.total}")
                print(f"Higher total: {winner.name.title()}")

        case "3":
            if not cache:
                print("Nothing saved yet.")
            for entry in sorted(cache.values(), key=lambda d: d["id"]):
                print(f"#{entry['id']:>4}  {entry['name']}")

        case "4":
            print("Gotta catch 'em all!")
            break

        case _:
            print("Invalid choice")

**Try this:** look up `pikachu`, then look it up again (you'll see `(from cache)`). Look up a nonsense name to test the error handling. Then stop your Wi-Fi and look up `pikachu` once more: it still works!

> 🔒 Add `pokedex_cache.json` to your `.gitignore`. It's data your program generates, not source code. (See the Developer Tools Guide.)

### Bonus ideas
- Show the Pokémon's **abilities** (explore `data["abilities"]`)
- Search by **type**
- Save the **favourite** Pokémon list
- Use your `Pokemon` class with another API

---
## 9. Practice and next steps

**🟢 Easy**
1. Fetch `ditto` and print its id, height and weight in sensible units.
2. Fetch the first 10 Pokémon names with `params={"limit": 10}`.

**🟡 Medium**
3. Write `get_types(name)` that returns a list of type names, handling every kind of error.
4. Compare the speed stat of two Pokémon chosen by the user.
5. Use `time.time()` to measure how much faster a cached lookup is than a fresh request.

**🔴 Challenge**
6. Pick **another free API without a key** (search for "free public APIs") and build a small program with it. Always begin by printing `response.json()` to understand its structure before you write any code.
7. Turn your project into a module: `pokedex_core.py` (the class and functions) and `main.py` (the menu).

### Remember
- Always use `timeout=`
- Always handle errors (`raise_for_status()` + `try/except`)
- Cache what you fetch; be polite to free services
- Never put **secret API keys** in code that you publish to GitHub

### ✅ You can now…
- explain what an API, a URL, JSON, and a status code are
- fetch and read JSON with `requests`
- pass `params=`, set timeouts, and handle failures properly
- cache results in a JSON file
- combine classes, files, exceptions and APIs in one project

**What's next →** Tkinter (building windows and buttons) and then databases with `sqlite3`.